# gesn-search: пересчёт на исправленной разметке и публикация на Hugging Face

1. Метрики всех подходов на разметке v2 (после доразметки) и парная значимость.
2. Модель → `<user>/e5-gesn` на Hugging Face.
3. Демо → Space `<user>/gesn-search` (Gradio, готовые векторы норм).

**Перед запуском:**
- **Add Input** → ноутбук с прошлым полным прогоном (`e5-gesn.zip`).
- **Add-ons → Secrets** → секрет `HF_TOKEN` (токен Hugging Face с правом Write), галочка «Attached».
- Session options: **GPU T4 x2**, Internet **On**.

Затем **Run All** (~10 минут).

In [ ]:
!rm -rf /tmp/gesn-search && git clone -q https://github.com/malma0/gesn-search /tmp/gesn-search
%cd /tmp/gesn-search
!pip install -q -e . --no-deps
!pip install -q "rank-bm25>=0.2.2" "pymorphy3>=2.0" "sentence-transformers>=5.0" gradio huggingface_hub

In [ ]:
import glob, zipfile

zips = glob.glob("/kaggle/input/**/e5-gesn.zip", recursive=True)
assert zips, "Не найден e5-gesn.zip — добавь вывод прошлого ноутбука через Add Input"
zipfile.ZipFile(zips[0]).extractall("models/e5-gesn")
print("модель:", zips[0])

## 1. Метрики на разметке v2

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --methods bm25 e5 hybrid ft=models/e5-gesn bm25+ft --out results/test_v2.md

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.analyze --methods bm25 e5 ft=models/e5-gesn --out-dir results/v2

## 2. Публикация модели и демо

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!CUDA_VISIBLE_DEVICES=0 python -m gesn.publish